# Yearly Average Popularity Calculator

This notebook calculates yearly average popularity from monthly popularity tables.

## Configuration

In [ ]:
import sys
import pathlib
sys.path.insert(0, str(pathlib.Path(".").resolve().parent.parent))

from config import DATA_DIR

# ============== CONFIGURATION ==============
# Year to calculate average for
YEAR = 2020

# Output: Where to save the yearly average table
INPUT_PATH = DATA_DIR / "cleaned_wikiPop_dumps"
OUTPUT_FILE = DATA_DIR / f"popularity_table_{YEAR}_avg.parquet"
# ============================================

## Step 1: Find Monthly Popularity Files

Search for all monthly popularity tables for the specified year.

In [2]:
# Find all monthly files for the year (exclude yearly avg files)
files = sorted([f for f in INPUT_PATH.glob(f"popularity_table_{YEAR}*.parquet") if "_avg" not in f.name])

print(f"Found {len(files)} monthly files for {YEAR}:")
for f in files:
    print(f"  - {f.name}")

Found 12 monthly files for 2023:
  - popularity_table_202301.parquet
  - popularity_table_202302.parquet
  - popularity_table_202303.parquet
  - popularity_table_202304.parquet
  - popularity_table_202305.parquet
  - popularity_table_202306.parquet
  - popularity_table_202307.parquet
  - popularity_table_202308.parquet
  - popularity_table_202309.parquet
  - popularity_table_202310.parquet
  - popularity_table_202311.parquet
  - popularity_table_202312.parquet


## Step 2: Load All Monthly Tables

Load each monthly file and extract the relevant columns (id, title, popularity).

In [3]:
import pandas as pd

# Load all monthly files
dfs = []
for f in files:
    df = pd.read_parquet(f)[["wikipedia_id", "wikipedia_title", "popularity", "rank"]]
    print(f"{f.name}: {len(df):,} articles")
    dfs.append(df)

# Combine all months
combined = pd.concat(dfs)
print(f"\nTotal rows (all months): {len(combined):,}")

popularity_table_202301.parquet: 5,903,530 articles
popularity_table_202302.parquet: 5,903,530 articles
popularity_table_202303.parquet: 5,903,530 articles
popularity_table_202304.parquet: 5,903,530 articles
popularity_table_202305.parquet: 5,903,530 articles
popularity_table_202306.parquet: 5,903,530 articles
popularity_table_202307.parquet: 5,903,530 articles
popularity_table_202308.parquet: 5,903,530 articles
popularity_table_202309.parquet: 5,903,530 articles
popularity_table_202310.parquet: 5,903,530 articles
popularity_table_202311.parquet: 5,903,530 articles
popularity_table_202312.parquet: 5,903,530 articles

Total rows (all months): 70,842,360


## Step 3: Calculate Average Popularity

Group by article (id, title) and calculate the mean popularity across all months.

In [4]:
# Calculate average popularity per article
result = combined.groupby(["wikipedia_id", "wikipedia_title"], as_index=False).agg({
    "popularity": "mean",
    "rank": "mean"
}).rename(columns={"popularity": "popularity_avg", "rank": "rank_avg"})

print(f"Unique articles: {len(result):,}")
result.head()

Unique articles: 5,903,530


,wikipedia_id,wikipedia_title,popularity_avg,rank_avg
0,1000,Hercule Poirot,84244.916667,1.016729e+04
1,10000,Eiffel,432.583333,1.180574e+06
2,10000001,Juan que reía,11.583333,4.668310e+06
3,10000009,La Noche del hurto,11.250000,4.665042e+06
4,1000001,NPU,1141.416667,6.910883e+05


## Step 4: Preview Results

Check the top and bottom articles by average popularity.

In [5]:
print("=== Top 20 Most Popular (Yearly Average) ===")
display(result.nlargest(20, "popularity_avg"))

print("\n=== Bottom 20 (Lowest Average Views) ===")
display(result.dropna(subset=["popularity_avg"]).nsmallest(20, "popularity_avg"))

=== Top 20 Most Popular (Yearly Average) ===


,wikipedia_id,wikipedia_title,popularity_avg,rank_avg
696082,15580374,Main Page,1.446738e+08,1.000000
5367045,60827,Cleopatra,5.365799e+06,4.416667
2876014,3524766,YouTube,4.438878e+06,10362.750000
3371875,39812824,2023 Cricket World Cup,3.215725e+06,186.333333
1131113,19326138,XXX: Return of Xander Cage,2.919775e+06,53.000000
424351,13260340,Indian Premier League,2.698472e+06,1916.708333
5611143,7529378,Facebook,2.594613e+06,23.500000
2501187,31591547,Instagram,2.320820e+06,20.250000
3286746,39034,J. Robert Oppenheimer,2.309642e+06,307.500000
4319211,49302020,XXX (film series),2.275362e+06,18.916667



=== Bottom 20 (Lowest Average Views) ===


,wikipedia_id,wikipedia_title,popularity_avg,rank_avg
22,1000007,Ettercap,1.0,5.702475e+06
516,10005467,The Patient,1.0,5.750280e+06
615,10006494,Mycobacterium massiliense,1.0,5.785127e+06
642,10006799,Tom (Lost),1.0,5.783968e+06
706,10007519,Super Week,1.0,5.783968e+06
708,10007553,Paragons (comics),1.0,5.755186e+06
1006,1001183,Scelsi,1.0,5.770700e+06
1565,10016761,TASCAR,1.0,5.780152e+06
1909,10020005,Tasimioidea,1.0,5.787485e+06
2006,10020836,Lexia (typeface),1.0,5.743038e+06


## Step 5: Save Output

Save the yearly average table to a parquet file.

In [6]:
import os

# Save to parquet
result.to_parquet(OUTPUT_FILE, index=False)
print(f"Saved to: {OUTPUT_FILE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE) / 1e6:.1f} MB")

Saved to: /Users/cyro/Documents/VSC/PopularityBias/scripts/../data/popularity_table_2023_avg.parquet
File size: 181.8 MB


## Summary Statistics

In [7]:
print(f"Year: {YEAR}")
print(f"Months included: {len(files)}")
print(f"Total articles: {len(result):,}")
print(f"\nPopularity statistics:")
result["popularity_avg"].describe()

Year: 2023
Months included: 12
Total articles: 5,903,530

Popularity statistics:


count    5.805609e+06
mean     1.116968e+03
std      6.079725e+04
min      1.000000e+00
25%      1.700000e+01
50%      6.483333e+01
75%      3.082500e+02
max      1.446738e+08
Name: popularity_avg, dtype: float64

In [10]:
from wikipedia_api import get_page_info

result = await get_page_info(id=19326138, days=2000)

print(result)  # Example Wikipedia ID

{'title': 'XXX: Return of Xander Cage', 'pageid': 19326138, 'url': 'https://en.wikipedia.org/wiki/XXX:_Return_of_Xander_Cage', 'summary': 'XXX: Return of Xander Cage (released as XXX: Reactivated in some countries) is a 2017 American action spy film directed by D.J. Caruso and written by F. Scott Frazier. The third installment in the XXX film series and a sequel to both XXX (2002) and XXX: State of the Union (2005), it stars Vin Diesel in the title role along with Donnie Yen, Deepika Padukone, Kris Wu, Ruby Rose, Tony Jaa, Nina Dobrev, Toni Collette, Ariadna Gutiérrez, Hermione Corfield, and Samuel L. Jackson.\nReleased on January 2...', 'pageviews': 79084361, 'period': '2020-08-04 to 2026-01-24'}
